In [0]:
from pyspark.sql.functions import (
    current_timestamp, lit, col, to_date,
    sum as spark_sum, current_date, to_timestamp
)
from pyspark.sql.window import Window

VOLUME_PATH = "/Volumes/de_workspace26/ecommerce_pawan/raw_files"
CATALOG     = "de_workspace26"
SCHEMA_B    = f"{CATALOG}.bronze_pawan"
SCHEMA_S    = f"{CATALOG}.silver_pawan"
SCHEMA_G    = f"{CATALOG}.gold_pawan"

print("Constants set.")
print("Volume path :", VOLUME_PATH)

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_G}")
print(f"✅ Schema created: {SCHEMA_G}")

In [0]:
spark.sql(f"""
    CREATE OR REPLACE VIEW {SCHEMA_G}.monthly_revenue_by_region AS
    SELECT
        YEAR(order_date)   AS year,
        MONTH(order_date)  AS month,
        region,
        SUM(revenue)       AS total_revenue,
        COUNT(order_id)    AS total_orders
    FROM {SCHEMA_S}.orders
    GROUP BY
        YEAR(order_date),
        MONTH(order_date),
        region
    ORDER BY year, month, region
""")

print(f"✅ View created: {SCHEMA_G}.monthly_revenue_by_region")

In [0]:
spark.sql(f"""
    SELECT *
    FROM   {SCHEMA_G}.monthly_revenue_by_region
""").show(20, truncate=False)

print("\nTotal rows in view:")
spark.sql(f"""
    SELECT COUNT(*) AS total_rows
    FROM   {SCHEMA_G}.monthly_revenue_by_region
""").show()

In [0]:
spark.sql(f"""
    CREATE OR REPLACE VIEW {SCHEMA_G}.top_products AS
    SELECT
        product_name,
        category,
        total_revenue,
        total_quantity,
        RANK() OVER (
            PARTITION BY category
            ORDER BY total_revenue DESC
        ) AS rank_in_category
    FROM (
        SELECT
            product_name,
            category,
            SUM(revenue)   AS total_revenue,
            SUM(quantity)  AS total_quantity
        FROM {SCHEMA_S}.orders
        GROUP BY product_name, category
    )
""")

print(f"✅ View created: {SCHEMA_G}.top_products")

In [0]:
print("=== All Products Ranked by Category ===")
spark.sql(f"""
    SELECT
        product_name,
        category,
        total_revenue,
        total_quantity,
        rank_in_category
    FROM   {SCHEMA_G}.top_products
    ORDER BY category, rank_in_category
""").show(20, truncate=False)

In [0]:
print("=== Top 10 Products by Total Revenue ===")
spark.sql(f"""
    SELECT
        product_name,
        category,
        total_revenue,
        total_quantity,
        rank_in_category
    FROM   {SCHEMA_G}.top_products
    ORDER BY total_revenue DESC
    LIMIT  10
""").show(truncate=False)

In [0]:
print("=" * 50)
print("GOLD VIEWS VERIFICATION")
print("=" * 50)

r1 = spark.sql(f"SELECT COUNT(*) AS cnt FROM {SCHEMA_G}.monthly_revenue_by_region").collect()[0]["cnt"]
r2 = spark.sql(f"SELECT COUNT(*) AS cnt FROM {SCHEMA_G}.top_products").collect()[0]["cnt"]

print(f"\n{SCHEMA_G}.monthly_revenue_by_region — {r1} rows")
print(f"{SCHEMA_G}.top_products               — {r2} rows")

print(f"\nRevenue by region summary:")
spark.sql(f"""
    SELECT
        region,
        SUM(total_revenue) AS total_revenue,
        SUM(total_orders)  AS total_orders
    FROM {SCHEMA_G}.monthly_revenue_by_region
    GROUP BY region
    ORDER BY total_revenue DESC
""").show()

print(f"Category-wise top product:")
spark.sql(f"""
    SELECT product_name, category, total_revenue
    FROM   {SCHEMA_G}.top_products
    WHERE  rank_in_category = 1
    ORDER BY total_revenue DESC
""").show()